# Autoencoder: Unsupervised Pattern Recognition — Jena Climate Dataset
**Task:** Anomaly detection · Latent space exploration · Weather pattern clustering  
**Builds on:** `output/processed_data.csv` (Stage 1) + `scaler.pkl` (Stage 2)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.cluster import KMeans

from utils.data_prep import load_and_split, scale_features, get_window_timestamps
from utils.sequencer import make_sequences
from utils.noise import add_gaussian_noise, signal_to_noise_ratio
from utils.models import build_conv_autoencoder, extract_encoder, extract_decoder
from utils.evaluator import (
    compute_reconstruction_mse,
    plot_reconstruction_examples,
    plot_noise_comparison,
    plot_training_loss,
    get_anomaly_mask,
    plot_anomaly_overlay,
    plot_latent_space,
    plot_elbow,
    plot_cluster_profiles,
    plot_cluster_timeline,
)

DATA_PATH    = Path('../output/processed_data.csv')
SCALER_PATH  = Path('../output/scaler.pkl')
AE_PATH      = Path('../output/autoencoder.h5')
ENC_PATH     = Path('../output/encoder.h5')

WINDOW_SIZE   = 24
NOISE_STD     = 0.1
BOTTLENECK    = 16

print('TF version:', tf.__version__)
plt.rcParams['figure.dpi'] = 100

---
## Section 1 — Data Preparation & Sequence Construction

In [ ]:
train_df, val_df, test_df = load_and_split(DATA_PATH)

In [ ]:
X_tr, X_va, X_te, feat_scaler, feature_cols = scale_features(
    train_df, val_df, test_df, scaler_path=SCALER_PATH
)
N_FEATURES = X_tr.shape[1]

# Index of Tpot (K) — used as the demo feature for reconstruction plots
# Tpot is interpretable and tightly correlated with temperature
DEMO_FEAT_IDX  = feature_cols.index('Tpot (K)')
DEMO_FEAT_NAME = 'Tpot (K)'
print(f'Demo feature: "{DEMO_FEAT_NAME}" at index {DEMO_FEAT_IDX}')

In [ ]:
X_train_seq = make_sequences(X_tr, WINDOW_SIZE)
X_val_seq   = make_sequences(X_va, WINDOW_SIZE)
X_test_seq  = make_sequences(X_te, WINDOW_SIZE)

print(f'X_train_seq: {X_train_seq.shape}')
print(f'X_val_seq  : {X_val_seq.shape}')
print(f'X_test_seq : {X_test_seq.shape}')

In [ ]:
# Datetime timestamp of the last timestep in each window — used for latent space colouring
train_ts = get_window_timestamps(train_df, WINDOW_SIZE)
val_ts   = get_window_timestamps(val_df,   WINDOW_SIZE)
test_ts  = get_window_timestamps(test_df,  WINDOW_SIZE)
all_ts   = train_ts.append(val_ts).append(test_ts)

print(f'Timestamps — train: {len(train_ts):,}  val: {len(val_ts):,}  test: {len(test_ts):,}')
print(f'Full range: {all_ts[0]} → {all_ts[-1]}')

In [ ]:
# T (degC) values aligned with each window (last timestep) — used for anomaly overlay
full_df = pd.concat([train_df, val_df, test_df]).sort_index()
T_all = full_df.loc[all_ts, 'T (degC)'].values

---
## Section 2 — Add Gaussian Noise

The autoencoder is trained on **clean data** (clean → clean reconstruction). Noise is added only to the **test set** to evaluate denoising capability after the fact.

In [ ]:
X_test_noisy = add_gaussian_noise(X_test_seq, std=NOISE_STD, seed=42)

snr = signal_to_noise_ratio(X_test_seq, X_test_noisy)
print(f'Mean SNR after adding σ={NOISE_STD} noise: {snr:.1f} dB')
print(f'Shapes — clean: {X_test_seq.shape}  noisy: {X_test_noisy.shape}')

In [ ]:
plot_noise_comparison(
    X_test_seq, X_test_noisy,
    feature_idx=DEMO_FEAT_IDX,
    feature_name=DEMO_FEAT_NAME,
    window_idx=0,
)

---
## Section 3 — 1D Convolutional Autoencoder

In [ ]:
autoencoder = build_conv_autoencoder(WINDOW_SIZE, N_FEATURES, bottleneck_dim=BOTTLENECK)
autoencoder.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(AE_PATH), monitor='val_loss', save_best_only=True, verbose=0
    ),
]

history = autoencoder.fit(
    X_train_seq, X_train_seq,          # input = output = clean windows
    validation_data=(X_val_seq, X_val_seq),
    epochs=200,
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
plot_training_loss(history)

---
## Section 4 — Denoising & Reconstruction Quality

In [ ]:
plot_reconstruction_examples(
    autoencoder, X_test_seq, X_test_noisy,
    feature_idx=DEMO_FEAT_IDX,
    feature_name=DEMO_FEAT_NAME,
    n_examples=4, seed=7,
)

In [ ]:
mse_test_clean = compute_reconstruction_mse(autoencoder, X_test_seq)
mse_test_noisy = compute_reconstruction_mse(autoencoder, X_test_noisy)

print(f'Mean reconstruction MSE — clean input : {mse_test_clean.mean():.6f}')
print(f'Mean reconstruction MSE — noisy input : {mse_test_noisy.mean():.6f}')
print(f'Noise penalty (ratio)                 : {mse_test_noisy.mean() / mse_test_clean.mean():.2f}×')

---
## Section 5 — Anomaly Detection via Reconstruction Error

Windows the autoencoder struggles to reconstruct likely correspond to unusual weather patterns. The 95th-percentile of **training** reconstruction error is used as the anomaly threshold.

In [ ]:
# Compute MSE for every window across all splits
X_all_seq = np.concatenate([X_train_seq, X_val_seq, X_test_seq], axis=0)
mse_all   = compute_reconstruction_mse(autoencoder, X_all_seq)
mse_train = mse_all[:len(X_train_seq)]

anomaly_mask, threshold = get_anomaly_mask(mse_all, mse_train, percentile=95)

n_anomalies = anomaly_mask.sum()
print(f'Anomaly threshold (p95 of train MSE) : {threshold:.6f}')
print(f'Total anomalies flagged              : {n_anomalies:,} / {len(mse_all):,} ({100*n_anomalies/len(mse_all):.1f}%)')

In [ ]:
plot_anomaly_overlay(all_ts, T_all, anomaly_mask, threshold, mse_all)

In [ ]:
# Inspect the 10 most anomalous windows
top_anomaly_idx = np.argsort(mse_all)[::-1][:10]
top_df = pd.DataFrame({
    'timestamp': all_ts[top_anomaly_idx],
    'reconstruction_mse': mse_all[top_anomaly_idx],
    'T (degC)': T_all[top_anomaly_idx],
}).reset_index(drop=True)
print('Top 10 most anomalous windows:')
print(top_df.to_string())

---
## Section 6 — Latent Space Exploration

The encoder compresses each 24-step window into a 16-dimensional bottleneck vector. PCA projects these to 2D for visualization.

In [ ]:
encoder = extract_encoder(autoencoder)
encoder.summary()
encoder.save(str(ENC_PATH))
print(f'Encoder saved → {ENC_PATH}')

In [ ]:
Z_all = encoder.predict(X_all_seq, batch_size=256, verbose=0)
print(f'Latent vectors shape: {Z_all.shape}  (n_windows × bottleneck_dim)')

In [ ]:
plot_latent_space(Z_all, all_ts, T_all)

---
## Section 7 — Weather Pattern Clustering

In [ ]:
# Elbow method on the full latent space
K_RANGE = range(2, 11)
inertias = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(Z_all)
    inertias.append(km.inertia_)

plot_elbow(inertias, K_RANGE)

In [ ]:
K_BEST = 4   # one cluster per season — adjust after inspecting elbow plot

kmeans = KMeans(n_clusters=K_BEST, random_state=42, n_init=10)
labels = kmeans.fit_predict(Z_all)
centroids = kmeans.cluster_centers_    # shape (K, 16)

for k in range(K_BEST):
    print(f'Cluster {k}: {(labels == k).sum():,} windows ({100*(labels==k).mean():.1f}%)')

In [ ]:
# Decode cluster centroids to get representative 24-step patterns
decoder = extract_decoder(autoencoder, bottleneck_dim=BOTTLENECK)
centroids_decoded = decoder.predict(centroids, verbose=0)  # shape (K, 24, n_features)
print(f'Decoded centroids shape: {centroids_decoded.shape}')

plot_cluster_profiles(
    centroids_decoded,
    feature_idx=DEMO_FEAT_IDX,
    feature_name=DEMO_FEAT_NAME,
)

In [ ]:
# Inspect mean T (degC) per cluster — helps label clusters
for k in range(K_BEST):
    mask = labels == k
    mean_T  = T_all[mask].mean()
    std_T   = T_all[mask].std()
    print(f'Cluster {k}: mean T = {mean_T:+.1f} °C  (±{std_T:.1f}), n={mask.sum():,}')

In [ ]:
plot_cluster_timeline(all_ts, labels, K=K_BEST)

---
## Section 8 — Save & Verify Artifacts

In [ ]:
for p in [AE_PATH, ENC_PATH, SCALER_PATH]:
    size = p.stat().st_size if p.exists() else -1
    status = f'OK  {size:,} bytes' if size >= 0 else 'MISSING'
    print(f'{p.name}: {status}')

In [ ]:
# Reload autoencoder and verify one forward pass
ae_reloaded = tf.keras.models.load_model(str(AE_PATH), compile=False)

X_recon = ae_reloaded.predict(X_test_seq[:1], verbose=0)
assert X_recon.shape == X_test_seq[:1].shape, \
    f'Shape mismatch: {X_recon.shape} vs {X_test_seq[:1].shape}'

final_mse = float(np.mean((X_test_seq - ae_reloaded.predict(X_test_seq, batch_size=256, verbose=0)) ** 2))
final_anomaly_rate = float(anomaly_mask.mean()) * 100

print(f'Final mean reconstruction MSE (test, clean) : {final_mse:.6f}')
print(f'Anomaly rate (full dataset, p95 threshold)  : {final_anomaly_rate:.1f}%')